# DST Pass-Through: Impact Dashboard
### YoY-Corrected Pre/Post Analysis

**Sections:**
1. Executive Summary — key metrics at a glance
2. Aggregated Pre/Post — bar charts comparing PRE vs POST for CY and LY
3. Weekly Event Study — time series with CY/LY overlay and phase shading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Rectangle
from pathlib import Path
from decimal import Decimal

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "0.92",
    "grid.linewidth": 0.8,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "figure.titlesize": 15,
    "figure.titleweight": "bold",
})

WINDOW_ORDER = ["PRE_LY", "POST_LY", "PRE_CY", "POST_CY"]
WINDOW_COLORS = {"PRE_LY": "#a8d8ea", "POST_LY": "#6baed6", "PRE_CY": "#fdcdac", "POST_CY": "#e6550d"}
WINDOW_LABELS = {"PRE_LY": "Pre (LY)", "POST_LY": "Post (LY)", "PRE_CY": "Pre (CY)", "POST_CY": "Post (CY)"}
PHASE_COLORS = {"Pre": "#b0c4de", "Post": "#abebc6"}


def to_float_df(df):
    """Convert Decimal columns to float."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            try:
                out[col] = out[col].apply(lambda x: float(x) if isinstance(x, (Decimal, int, float)) else x)
            except (ValueError, TypeError):
                pass
        try:
            out[col] = pd.to_numeric(out[col], errors="ignore")
        except Exception:
            pass
    return out


def load_table(table_name):
    """Read a materialized DST table from Databricks."""
    return to_float_df(spark.sql(f"SELECT * FROM production.supply_analytics.{table_name}").toPandas())


def load_weekly_sql(sql_path, params_override=None):
    """Read and run a weekly SQL file, optionally overriding hard-coded params."""
    sql_text = Path(sql_path).read_text()
    if params_override:
        for k, v in params_override.items():
            sql_text = sql_text.replace(f"params.{k}", f"'{v}'" if isinstance(v, str) else str(v))
    return to_float_df(spark.sql(sql_text).toPandas())


def did_pct(df, metric, pre_label="Pre", post_label="Post", period_col="period", phase_col="phase"):
    """Compute YoY-corrected DID % change for a single metric."""
    pivot = df.pivot_table(index=phase_col, columns=period_col, values=metric, aggfunc="mean")
    if "CY" not in pivot.columns or "LY" not in pivot.columns:
        return np.nan
    cy_pre = pivot.loc[pre_label, "CY"] if pre_label in pivot.index else np.nan
    cy_post = pivot.loc[post_label, "CY"] if post_label in pivot.index else np.nan
    ly_pre = pivot.loc[pre_label, "LY"] if pre_label in pivot.index else np.nan
    ly_post = pivot.loc[post_label, "LY"] if post_label in pivot.index else np.nan
    if any(pd.isna([cy_pre, cy_post, ly_pre, ly_post])):
        return np.nan
    eps = 1e-12
    return (cy_post - cy_pre) / (cy_pre + eps) - (ly_post - ly_pre) / (ly_pre + eps)


print("Setup complete.")

Setup complete.


## 1. Load Data
Run the `Aggregated_final/*.dbquery.ipynb` notebooks first to materialize the `dst_*` tables, then load here.

In [ ]:
# --- Aggregated data (from materialized tables) ---
bookings_agg = load_table("dst_booking_dataset")
supply_agg = load_table("dst_supply_dataset")
customer_agg = load_table("dst_session_performance_dataset")
price_agg = load_table("dst_price_change")
cancel_agg = load_table("dst_cancellation_dataset")
parity_agg = load_table("dst_price_parity")

print(f"Loaded aggregated tables:")
for name, df in [("Bookings", bookings_agg), ("Supply", supply_agg), ("Customer", customer_agg),
                  ("Price", price_agg), ("Cancellation", cancel_agg), ("Parity", parity_agg)]:
    print(f"  {name}: {len(df)} rows, columns: {list(df.columns)[:6]}...")

In [ ]:
# --- Weekly data (run SQL directly) ---
def _resolve_weekly(name):
    for base in [
        Path.cwd() / "DST_Analysis" / "SQL" / "weekly",
        Path.cwd().parent / "SQL" / "weekly",
        Path.cwd() / "SQL" / "weekly",
        Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/weekly"),
    ]:
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find {name}")

weekly_bookings = to_float_df(spark.sql(_resolve_weekly("weekly_bookings.sql").read_text()).toPandas())
weekly_supply = to_float_df(spark.sql(_resolve_weekly("weekly_supplier.sql").read_text()).toPandas())
weekly_customer = to_float_df(spark.sql(_resolve_weekly("weekly_customer_data.sql").read_text()).toPandas())
weekly_price = to_float_df(spark.sql(_resolve_weekly("weekly_price_change.sql").read_text()).toPandas())
weekly_cancel = to_float_df(spark.sql(_resolve_weekly("weekly_cancellation.sql").read_text()).toPandas())
weekly_parity = to_float_df(spark.sql(_resolve_weekly("weekly_price_parity.sql").read_text()).toPandas())

print(f"Loaded weekly data:")
for name, df in [("Bookings", weekly_bookings), ("Supply", weekly_supply), ("Customer", weekly_customer),
                  ("Price", weekly_price), ("Cancellation", weekly_cancel), ("Parity", weekly_parity)]:
    print(f"  {name}: {len(df)} rows")

In [ ]:
def plot_agg_bars(df, metrics, labels, title, fmt=",.0f", pct=False, figsize=(14, 5)):
    """Grouped bar chart: PRE_LY / POST_LY / PRE_CY / POST_CY for each metric."""
    ordered = [w for w in WINDOW_ORDER if w in df["period_window"].values]
    sub = df[df["period_window"].isin(ordered)].set_index("period_window").reindex(ordered)

    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=False)
    if n == 1:
        axes = [axes]

    for ax, metric, label in zip(axes, metrics, labels):
        vals = [sub.loc[w, metric] if w in sub.index and pd.notna(sub.loc[w, metric]) else 0 for w in ordered]
        colors = [WINDOW_COLORS[w] for w in ordered]
        bars = ax.bar(range(len(ordered)), vals, color=colors, edgecolor="white", linewidth=0.8)
        ax.set_xticks(range(len(ordered)))
        ax.set_xticklabels([WINDOW_LABELS[w] for w in ordered], fontsize=9, rotation=15)
        ax.set_title(label, fontsize=11)
        if pct:
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))
        for bar, v in zip(bars, vals):
            txt = f"{v:{fmt}}" if not pct else f"{v:.1%}"
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    txt, ha="center", va="bottom", fontsize=8, fontweight="bold")

    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    return fig


def plot_weekly_event_study(df, cy_col, ly_col, title, ylabel,
                            week_col="week_index", period_col="period", phase_col="phase",
                            pct=False, figsize=(12, 4)):
    """Line chart: CY vs LY by week_index with Pre/Post shading."""
    cy = df[df[period_col] == "CY"].sort_values(week_col)
    ly = df[df[period_col] == "LY"].sort_values(week_col)

    fig, ax = plt.subplots(figsize=figsize)

    for phase_name, color in PHASE_COLORS.items():
        phase_rows = df[df[phase_col] == phase_name]
        if not phase_rows.empty:
            wmin, wmax = phase_rows[week_col].min(), phase_rows[week_col].max()
            ax.axvspan(wmin - 0.4, wmax + 0.4, alpha=0.18, color=color, label=f"{phase_name} window")

    ax.plot(cy[week_col], cy[cy_col], "o-", color="#e6550d", linewidth=2, markersize=5, label="CY", zorder=3)
    ax.plot(ly[week_col], ly[ly_col], "s--", color="#6baed6", linewidth=1.5, markersize=4, label="LY", zorder=3, alpha=0.8)
    ax.axvline(0, color="red", linestyle=":", linewidth=1.5, alpha=0.7, label="Event date")

    if pct:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

    ax.set_xlabel("Week offset from event", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)
    fig.tight_layout()
    return fig


def plot_weekly_did(df, cy_col, ly_col, title, ylabel,
                    week_col="week_index", period_col="period", phase_col="phase",
                    figsize=(12, 3.5)):
    """YoY-adjusted DID line: (CY/mean_PRE_CY - 1) - (LY/mean_PRE_LY - 1)."""
    cy = df[df[period_col] == "CY"].sort_values(week_col)
    ly = df[df[period_col] == "LY"].sort_values(week_col)

    merged = cy[[week_col, phase_col, cy_col]].merge(
        ly[[week_col, ly_col]], on=week_col, how="inner"
    )
    pre = merged[merged[phase_col] == "Pre"]
    cy_base = pre[cy_col].mean()
    ly_base = pre[ly_col].mean()
    eps = 1e-12
    merged["did"] = (merged[cy_col] / (cy_base + eps) - 1) - (merged[ly_col] / (ly_base + eps) - 1)

    fig, ax = plt.subplots(figsize=figsize)
    for phase_name, color in PHASE_COLORS.items():
        phase_rows = merged[merged[phase_col] == phase_name]
        if not phase_rows.empty:
            wmin, wmax = phase_rows[week_col].min(), phase_rows[week_col].max()
            ax.axvspan(wmin - 0.4, wmax + 0.4, alpha=0.18, color=color)

    ax.axhline(0, color="red", linestyle="--", linewidth=1.2)
    ax.plot(merged[week_col], merged["did"], "o-", color="black", linewidth=1.5, markersize=4)
    ax.axvline(0, color="red", linestyle=":", linewidth=1.2, alpha=0.6)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))
    ax.set_xlabel("Week offset from event")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    return fig


print("Chart helpers ready.")

---
## 1. Executive Summary
Key YoY-corrected DID metrics at a glance — does the CY Pre→Post change differ from the LY Pre→Post change?

In [ ]:
def compute_agg_did(df, metric, window_col="period_window"):
    """DID from aggregated PRE_LY/POST_LY/PRE_CY/POST_CY rows."""
    vals = df.set_index(window_col)[metric]
    required = ["PRE_LY", "POST_LY", "PRE_CY", "POST_CY"]
    if not all(w in vals.index for w in required):
        return np.nan
    eps = 1e-12
    cy_pct = (vals["POST_CY"] - vals["PRE_CY"]) / (vals["PRE_CY"] + eps)
    ly_pct = (vals["POST_LY"] - vals["PRE_LY"]) / (vals["PRE_LY"] + eps)
    return cy_pct - ly_pct


summary_rows = []
for label, df, metric in [
    ("Net Revenue", bookings_agg, "nr"),
    ("GMV", bookings_agg, "gmv"),
    ("Bookings", bookings_agg, "bookings"),
    ("Active Tours", supply_agg, "active_tours"),
    ("Active Suppliers", supply_agg, "active_suppliers"),
    ("Share Active Tours", supply_agg, "share_active_tours"),
    ("Conversion Rate", customer_agg, "conversion_rate"),
    ("Add-to-Cart Rate", customer_agg, "add_to_cart_rate"),
    ("Cancellation Rate", cancel_agg, "cancellation_rate"),
    ("Overall Parity", parity_agg, "overall_parity_rate"),
]:
    did = compute_agg_did(df, metric)
    vals = df.set_index("period_window")[metric]
    pre_cy = vals.get("PRE_CY", np.nan)
    post_cy = vals.get("POST_CY", np.nan)
    summary_rows.append({
        "Metric": label,
        "Pre (CY)": pre_cy,
        "Post (CY)": post_cy,
        "CY Change %": (post_cy - pre_cy) / (pre_cy + 1e-12) if pd.notna(pre_cy) else np.nan,
        "DID % (YoY-corrected)": did,
    })

summary_df = pd.DataFrame(summary_rows)

def color_did(val):
    if pd.isna(val):
        return ""
    if abs(val) < 0.005:
        return "background-color: #f0f0f0"
    return "background-color: #d4edda" if val > 0 else "background-color: #f8d7da"

display(summary_df.style.format({
    "Pre (CY)": "{:,.2f}",
    "Post (CY)": "{:,.2f}",
    "CY Change %": "{:+.2%}",
    "DID % (YoY-corrected)": "{:+.2%}",
}, na_rep="—").applymap(color_did, subset=["DID % (YoY-corrected)"]).set_caption(
    "Executive Summary: YoY-Corrected DID — Green = positive, Red = negative"
))

---
## 2. Aggregated Pre/Post Analysis
Comparing **PRE vs POST** windows for both **Current Year (CY)** and **Last Year (LY)** to isolate DST-driven changes from seasonality.

In [ ]:
# --- 2.1 Bookings & Net Revenue ---
fig = plot_agg_bars(
    bookings_agg,
    metrics=["nr", "gmv", "bookings", "tickets"],
    labels=["Net Revenue", "GMV", "Bookings", "Tickets"],
    title="Bookings & Revenue — Pre vs Post",
)
plt.show()

fig2 = plot_agg_bars(
    bookings_agg,
    metrics=["avg_nr_per_booking", "avg_gmv_per_ticket"],
    labels=["Avg NR / Booking", "Avg GMV / Ticket"],
    title="Unit Economics — Pre vs Post",
    figsize=(10, 4),
)
plt.show()

In [ ]:
# --- 2.2 Supply Continuity ---
fig = plot_agg_bars(
    supply_agg,
    metrics=["active_tours", "active_suppliers", "total_tours", "total_suppliers"],
    labels=["Active Tours", "Active Suppliers", "Total Tours", "Total Suppliers"],
    title="Supply Continuity — Pre vs Post",
)
plt.show()

fig2 = plot_agg_bars(
    supply_agg,
    metrics=["share_active_tours", "share_active_suppliers"],
    labels=["Share Active Tours", "Share Active Suppliers"],
    title="Share of Active Supply — Pre vs Post",
    pct=True,
    figsize=(10, 4),
)
plt.show()

fig3 = plot_agg_bars(
    supply_agg,
    metrics=["avg_days_online_per_tour_per_week", "avg_days_online_per_supplier_per_week"],
    labels=["Avg Days Online / Tour / Week", "Avg Days Online / Supplier / Week"],
    title="Intensive Margin — Days Online Per Week",
    fmt=".1f",
    figsize=(10, 4),
)
plt.show()

In [ ]:
# --- 2.3 Customer Metrics ---
fig = plot_agg_bars(
    customer_agg,
    metrics=["conversion_rate", "click_through_rate", "add_to_cart_rate"],
    labels=["Conversion Rate", "Click-Through Rate", "Add-to-Cart Rate"],
    title="Customer Funnel — Pre vs Post",
    pct=True,
)
plt.show()

fig2 = plot_agg_bars(
    customer_agg,
    metrics=["bookings", "visitors", "impressions"],
    labels=["Bookings", "Visitors", "Impressions"],
    title="Customer Volume — Pre vs Post",
)
plt.show()

In [ ]:
# --- 2.4 Price Change Analysis ---
# Show balanced panel medians and % of tours/suppliers that changed price
price_display = price_agg[["country_name", "period_window", "total_tours", "total_suppliers",
                            "balanced_median_price_3m", "balanced_median_price_6m", "balanced_median_price_12m",
                            "pct_tours_changed_3m", "pct_tours_changed_6m", "pct_tours_changed_12m",
                            "pct_suppliers_changed_3m", "pct_suppliers_changed_6m", "pct_suppliers_changed_12m"]].copy()
price_display = price_display.sort_values(["country_name", "period_window"],
    key=lambda s: s.map({"PRE_LY": 0, "POST_LY": 1, "PRE_CY": 2, "POST_CY": 3}) if s.name == "period_window" else s)
print("Price Change Analysis — Balanced Panel (same tours in all 4 windows):")
display(price_display.style.format({
    "balanced_median_price_3m": "€{:.2f}", "balanced_median_price_6m": "€{:.2f}", "balanced_median_price_12m": "€{:.2f}",
    "pct_tours_changed_3m": "{:.1f}%", "pct_tours_changed_6m": "{:.1f}%", "pct_tours_changed_12m": "{:.1f}%",
    "pct_suppliers_changed_3m": "{:.1f}%", "pct_suppliers_changed_6m": "{:.1f}%", "pct_suppliers_changed_12m": "{:.1f}%",
}, na_rep="—").set_caption("Balanced Panel: Median Prices & % Changed (>1% threshold)"))

In [ ]:
# --- 2.5 Cancellations ---
fig = plot_agg_bars(
    cancel_agg,
    metrics=["cancellation_rate", "cancellation_rate_3m", "cancellation_rate_6m"],
    labels=["Overall Cancel Rate", "Cancel Rate (3M travel)", "Cancel Rate (6M travel)"],
    title="Cancellation Rates — Pre vs Post",
    pct=True,
)
plt.show()

fig2 = plot_agg_bars(
    cancel_agg,
    metrics=["cancellation_rate_nr", "cancellation_rate_nr_3m", "cancellation_rate_nr_6m"],
    labels=["NR Cancel Rate", "NR Cancel Rate (3M)", "NR Cancel Rate (6M)"],
    title="NR-Weighted Cancellation Rates — Pre vs Post",
    pct=True,
)
plt.show()

In [ ]:
# --- 2.6 Price Parity ---
fig = plot_agg_bars(
    parity_agg,
    metrics=["overall_parity_rate", "supplier_parity_rate", "viator_parity_rate", "tiqets_parity_rate"],
    labels=["Overall Parity", "vs Supplier Direct", "vs Viator", "vs Tiqets"],
    title="Price Parity Rates (impression-weighted) — Pre vs Post",
    pct=True,
)
plt.show()

---
## 3. Weekly Event Study
Each chart shows CY vs LY by week, with **Pre** (blue) and **Post** (green) shading.  
Week 0 = event date. Negative weeks = pre-period. DID charts show the YoY-adjusted difference indexed to the pre-period mean.

In [ ]:
# --- 3.1 Weekly Bookings & NR ---
for metric, label in [("nr", "Net Revenue"), ("gmv", "GMV"), ("bookings", "Bookings")]:
    fig = plot_weekly_event_study(weekly_bookings, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label)
    plt.show()

    fig2 = plot_weekly_did(weekly_bookings, cy_col=metric, ly_col=metric,
                           title=f"Weekly {label}: YoY-Adjusted DID (indexed to Pre mean)",
                           ylabel="DID % change")
    plt.show()

In [ ]:
# --- 3.2 Weekly Supply ---
for metric, label in [("active_tours", "Active Tours"), ("active_suppliers", "Active Suppliers"),
                       ("share_active_tours", "Share Active Tours")]:
    fig = plot_weekly_event_study(weekly_supply, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label,
                                  pct=(metric.startswith("share")))
    plt.show()

fig = plot_weekly_did(weekly_supply, cy_col="active_tours", ly_col="active_tours",
                      title="Active Tours: YoY-Adjusted DID", ylabel="DID % change")
plt.show()

# Avg days online per tour
fig = plot_weekly_event_study(weekly_supply, cy_col="avg_days_online_per_tour_per_week",
                              ly_col="avg_days_online_per_tour_per_week",
                              title="Avg Days Online / Tour / Week: CY vs LY",
                              ylabel="Days online")
plt.show()

In [ ]:
# --- 3.3 Weekly Customer Funnel ---
for metric, label, is_pct in [
    ("conversion_rate", "Conversion Rate", True),
    ("add_to_cart_rate", "Add-to-Cart Rate", True),
    ("click_through_rate", "Click-Through Rate", True),
    ("bookings_per_click", "Bookings per Click", False),
]:
    fig = plot_weekly_event_study(weekly_customer, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label, pct=is_pct)
    plt.show()

In [ ]:
# --- 3.4 Weekly Price Change ---
for metric, label in [
    ("median_from_red_price", "Median From-Price (Red)"),
    ("median_final_red_3m", "Median Final Red Price (3M travel)"),
    ("median_final_red_6m", "Median Final Red Price (6M travel)"),
    ("median_timeslots_3m", "Median Timeslots (3M travel)"),
]:
    fig = plot_weekly_event_study(weekly_price, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label)
    plt.show()

In [ ]:
# --- 3.5 Weekly Cancellations ---
for metric, label, is_pct in [
    ("cancellation_rate", "Cancellation Rate (Overall)", True),
    ("cancellation_rate_3m", "Cancellation Rate (3M travel)", True),
    ("cancellation_rate_nr", "NR Cancellation Rate", True),
]:
    fig = plot_weekly_event_study(weekly_cancel, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label, pct=is_pct)
    plt.show()

In [ ]:
# --- 3.6 Weekly Price Parity ---
for metric, label in [
    ("overall_parity_rate", "Overall Parity Rate"),
    ("supplier_parity_rate", "Supplier Direct Parity"),
    ("viator_parity_rate", "Viator Parity"),
    ("tiqets_parity_rate", "Tiqets Parity"),
]:
    fig = plot_weekly_event_study(weekly_parity, cy_col=metric, ly_col=metric,
                                  title=f"Weekly {label}: CY vs LY", ylabel=label, pct=True)
    plt.show()

---
**HTML Export:** Run `generate_html_dashboard.py` to produce `DST_Dashboard.html` — a self-contained file you can share without Python or Databricks.

In [ ]:
# %run ./generate_html_dashboard